In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w = Window.partitionBy("event_id").orderBy(F.col("_ingested_at").desc())

clean = (spark.table("workspace.bronze.events")
    .select(
        "event_id",
        F.col("customer_id").cast("int").alias("customer_id"),
        F.lower(F.trim("event_type")).alias("event_type"),
        F.to_timestamp("event_ts").alias("event_ts"),
        F.get_json_object("device", "$.type").alias("device_type"),      # flatten JSON string fields
        F.get_json_object("device", "$.os").alias("device_os"),
        F.get_json_object("page", "$.url").alias("page_url"),
        F.coalesce(F.get_json_object("page", "$.referrer"), F.lit("unknown")).alias("referrer"),
        "_ingested_at")
    .withColumn("product_id", F.regexp_extract("page_url", r"/product/(\d+)", 1).cast("int"))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1").drop("rn"))

is_valid = (
    F.col("event_type").isin("page_view", "add_to_cart", "purchase")
    & F.col("event_ts").isNotNull()
    & F.col("customer_id").isNotNull()
)

clean.filter(is_valid).createOrReplaceTempView("events_updates")
(clean.filter(~is_valid).write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable("workspace.silver.events_quarantine"))

spark.sql("CREATE TABLE IF NOT EXISTS workspace.silver.events AS SELECT * FROM events_updates WHERE 1=0")

spark.sql("""
  MERGE INTO workspace.silver.events t
  USING events_updates s ON t.event_id = s.event_id
  WHEN MATCHED THEN UPDATE SET *
  WHEN NOT MATCHED THEN INSERT *
""")

print("silver.events:", spark.table("workspace.silver.events").count())
print("quarantined:", spark.table("workspace.silver.events_quarantine").count())